In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score


import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = OneHotEncoder(sparse_output=False)

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [7]:
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats.distributions as dists

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size = 0.3, random_state = 0)

In [8]:
space_lgbm={'colsample_bytree': dists.uniform(0, 1),
            'drop_rate': dists.uniform(0, 1),
            'learning_rate': dists.uniform(0, 1),
            'max_bin': numpy.arange(1, 1000),
            'max_depth': numpy.arange(1, 1000),
            'max_drop': numpy.arange(1, 1000),
            'min_child_samples': numpy.arange(1, 1000),
            'min_data_in_leaf': numpy.arange(1, 1000),
            'num_leaves': numpy.arange(1, 1000),
            'n_estimators': numpy.arange(100, 10000),
            'reg_alpha': dists.uniform(0, 1),
            'reg_lambda': dists.uniform(0, 1),
            'skip_drop': dists.uniform(0, 1),
            'verbosity': [-1]
           }

print(space_lgbm['n_estimators'])

lgbm=LGBMRegressor()

lgbm_cv = RandomizedSearchCV(lgbm, param_distributions=space_lgbm, cv=10, verbose=3)
lgbm_cv.fit(X_train, y_train)

lgbm_hyperparams=lgbm_cv.best_params_

print("Tuned Decision Tree Parameters: {}".format(lgbm_cv.best_params_))
print("Best score is {}".format(lgbm_cv.best_score_))

[ 100  101  102 ... 9997 9998 9999]
Fitting 10 folds for each of 10 candidates, totalling 100 fits
[CV 1/10] END colsample_bytree=0.3765136546575276, drop_rate=0.8590084425127401, learning_rate=0.9238500251826626, max_bin=62, max_depth=945, max_drop=685, min_child_samples=337, min_data_in_leaf=14, n_estimators=3816, num_leaves=809, reg_alpha=0.39389681327953596, reg_lambda=0.9657118589074224, skip_drop=0.20178489422661394, verbosity=-1;, score=-0.353 total time=  21.4s
[CV 2/10] END colsample_bytree=0.3765136546575276, drop_rate=0.8590084425127401, learning_rate=0.9238500251826626, max_bin=62, max_depth=945, max_drop=685, min_child_samples=337, min_data_in_leaf=14, n_estimators=3816, num_leaves=809, reg_alpha=0.39389681327953596, reg_lambda=0.9657118589074224, skip_drop=0.20178489422661394, verbosity=-1;, score=-0.469 total time=  15.4s
[CV 3/10] END colsample_bytree=0.3765136546575276, drop_rate=0.8590084425127401, learning_rate=0.9238500251826626, max_bin=62, max_depth=945, max_drop=

lightgbm = LGBMRegressor(colsample_bytree=lgbm_hyperparams['colsample_bytree'], 
                         drop_rate=lgbm_hyperparams['drop_rate'],
                         learning_rate=lgbm_hyperparams['learning_rate'],
                         max_bin=lgbm_hyperparams['max_bin'], 
                         max_depth=int(lgbm_hyperparams['max_depth']),
                         max_drop=lgbm_hyperparams['max_drop']
                         min_child_samples=lgbm_hyperparams['min_child_samples'],
                         min_data_in_leaf=lgbm_hyperparams['min_data_in_leaf'],
                         n_estimators=lgbm_hyperparams['n_estimators'], 
                         num_leaves=lgbm_hyperparams['num_leaves'], 
                         reg_alpha=lgbm_hyperparams['reg_alpha'],
                         reg_lambda=lgbm_hyperparams['reg_lambda'],
                         skip_drop=lgbm_hyperparams['skip_drop']
                         verbosity=-1)space={ 'n_estimators': numpy.arange(1000, 20000),
        'max_depth': numpy.arange(3, 30),
        'gamma': numpy.arange(1,9),
        'reg_alpha' : numpy.arange(40,180),
        'reg_lambda' : dists.uniform(0, 1),
        'colsample_bytree' : dists.uniform(0.5, 1),
        'min_child_weight' : numpy.arange(0, 10),
        'subsample' : dists.uniform(0, 1),
        'learning_rate': dists.uniform(0, 1)
    }
print(space['n_estimators'])

XGB=XGBRegressor()

tree_cv = RandomizedSearchCV(XGB, param_distributions=space, cv=10, verbose=3)
tree_cv.fit(X_train, y_train)

XGB_hyperparams=tree_cv.best_params_
print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
print("Best score is {}".format(tree_cv.best_score_))

In [9]:
gbr = GradientBoostingRegressor(n_estimators=3000, 
                                learning_rate=0.05, 
                                max_depth=4, 
                                max_features='sqrt', 
                                min_samples_leaf=15, 
                                min_samples_split=10, 
                                loss='huber', 
                                random_state =42)    

lightgbm = LGBMRegressor(colsample_bytree=lgbm_hyperparams['colsample_bytree'], 
                         drop_rate=lgbm_hyperparams['drop_rate'],
                         learning_rate=lgbm_hyperparams['learning_rate'],
                         max_bin=lgbm_hyperparams['max_bin'], 
                         max_depth=int(lgbm_hyperparams['max_depth']),
                         max_drop=lgbm_hyperparams['max_drop'],
                         min_child_samples=lgbm_hyperparams['min_child_samples'],
                         min_data_in_leaf=lgbm_hyperparams['min_data_in_leaf'],
                         n_estimators=lgbm_hyperparams['n_estimators'], 
                         num_leaves=lgbm_hyperparams['num_leaves'], 
                         reg_alpha=lgbm_hyperparams['reg_alpha'],
                         reg_lambda=lgbm_hyperparams['reg_lambda'],
                         skip_drop=lgbm_hyperparams['skip_drop'],
                         verbosity=-1)

"""xgboost = XGBRegressor(n_estimators=int(XGB_hyperparams['n_estimators']),  
                       learning_rate=XGB_hyperparams['learning_rate'],
                       colsample_bytree=XGB_hyperparams['colsample_bytree'], 
                       subsample=XGB_hyperparams['subsample'], 
                       objective='binary:logistic',
                       min_child_weight=XGB_hyperparams['min_child_weight'],
                       gamma=XGB_hyperparams['gamma'],
                       max_depth=int(XGB_hyperparams['max_depth']),
                       reg_alpha=XGB_hyperparams['reg_alpha'],
                       reg_lambda=XGB_hyperparams['reg_lambda'])"""
#reg:logistic
#binary:logitraw
#binary:logistic

catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=3.5,
                             min_child_samples=32,
                             grow_policy='Depthwise',
                             iterations=8000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

In [10]:
#model = StackingRegressor(estimators=[('catboost', catboost), ('gbr', gbr),
#                                      ('xgboost', xgboost), ('lightgbm', lightgbm)],
#                                      final_estimator=svr)
model=lightgbm

In [11]:
model.fit(train_x, train_y)

LGBMRegressor(colsample_bytree=0.09354180816918023,
              drop_rate=0.6082064069343034, learning_rate=0.17895320272837667,
              max_bin=896, max_depth=552, max_drop=429, min_child_samples=883,
              min_data_in_leaf=685, n_estimators=2059, num_leaves=749,
              reg_alpha=0.7871002992889985, reg_lambda=0.34479003468373615,
              skip_drop=0.6887041025248984, verbosity=-1)

In [12]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [13]:
test_predictions = model.predict(test)

In [14]:
test_predictions

array([0.11058014, 0.6377495 , 0.1553746 ])

In [15]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.110580
1,28801,0.637749
2,28802,0.155375


In [16]:
submission.to_csv('submission.csv', index = False)